# 🛡️ Cybersecurity AI Assistant — Fine-Tuning Llama-2-7B with Unsloth

### ✅ Fixes applied:
- `MAX_SEQ_LENGTH` : 4096 → **2048** (fixes OOM)
- `BATCH_SIZE` : 2 → **1** (fixes OOM)
- `GRAD_ACCUM_STEPS` : 4 → **8** (keeps effective batch size = 8)
- `total_mem` → **total_memory** (fixes AttributeError)
- **Resume from checkpoint-800** ✅

---

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/cybersec_train_5000.jsonl"

import os
if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found: {DATASET_PATH}")
    size_mb = os.path.getsize(DATASET_PATH) / (1024 * 1024)
    print(f"   Size: {size_mb:.1f} MB")
else:
    print(f"❌ Dataset NOT found at: {DATASET_PATH}")

Mounted at /content/drive
✅ Dataset found: /content/drive/MyDrive/cybersec_train_5000.jsonl
   Size: 18.9 MB


## Step 3: Load the Model with Unsloth
> ✅ FIX: `MAX_SEQ_LENGTH` = **2048** (was 4096)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME     = "unsloth/llama-2-7b-bnb-4bit"
MAX_SEQ_LENGTH = 2048   # ✅ FIX: was 4096
DTYPE          = None
LOAD_IN_4BIT   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Max sequence length: {MAX_SEQ_LENGTH}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.87G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


✅ Model loaded: unsloth/llama-2-7b-bnb-4bit
   Max sequence length: 2048


## Step 4: Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA adapters added")
model.print_trainable_parameters()

Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapters added
trainable params: 39,976,960 || all params: 6,778,392,576 || trainable%: 0.5898


## Step 5: Load and Prepare the Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(f"✅ Dataset loaded: {len(dataset)} training examples")
print(f"\n📋 Sample entry (first 500 chars):")
print(dataset[0]["text"][:500])
print("...")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded: 5000 training examples

📋 Sample entry (first 500 chars):
<s>[INST] <<SYS>>
You are a highly specialized AI assistant for advanced cyber-defense whose mission is to deliver accurate, in-depth, actionable guidance on information-security principles—confidentiality, integrity, availability, authenticity, non-repudiation, and privacy—by offering concise executive summaries that drill down into technical detail, industry standards, and threat models while referencing frameworks such as NIST CSF and MITRE ATT&CK; you may share defensive scripts, detection r
...


## Step 6: Configure Training
> ✅ FIX: `BATCH_SIZE` = **1** | `GRAD_ACCUM_STEPS` = **8** → effective batch size = 8

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR       = "/content/drive/MyDrive/cybersec-llama2-finetuned"
NUM_EPOCHS       = 3
BATCH_SIZE       = 1    # ✅ FIX: was 2
GRAD_ACCUM_STEPS = 8    # ✅ FIX: was 4
LEARNING_RATE    = 2e-4
WARMUP_STEPS     = 50
SAVE_STEPS       = 200
LOGGING_STEPS    = 25

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        save_steps=SAVE_STEPS,
        logging_steps=LOGGING_STEPS,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        save_total_limit=3,
        report_to="none",
    ),
)

print("✅ Trainer configured")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Output: {OUTPUT_DIR}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

✅ Trainer configured
   Epochs: 3
   Effective batch size: 8
   Learning rate: 0.0002
   Output: /content/drive/MyDrive/cybersec-llama2-finetuned


## Step 7: Check GPU Memory Before Training
> ✅ FIX: `total_mem` → `total_memory`

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)  # ✅ FIX

print(f"🖥️  GPU: {gpu_stats.name}")
print(f"   Total VRAM: {max_memory} GB")
print(f"   Currently reserved: {start_gpu_memory} GB")
print(f"   Available for training: {max_memory - start_gpu_memory:.1f} GB")

🖥️  GPU: Tesla T4
   Total VRAM: 14.563 GB
   Currently reserved: 3.781 GB
   Available for training: 10.8 GB


## Step 8: 🚀 Start Training — Resume from checkpoint-800
> ✅ يكمل من **checkpoint-800** — باقيك غير **1075 steps** من أصل 1875

In [ ]:
import os
import torch

# ✅ Clean VRAM
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ✅ Resume from checkpoint-800
CHECKPOINT_PATH = "/content/drive/MyDrive/cybersec-llama2-finetuned/checkpoint-800"

if os.path.exists(CHECKPOINT_PATH):
    print(f"🔁 Resuming from: checkpoint-800")
    print(f"   Remaining steps: ~1075 / 1875")
    resume = CHECKPOINT_PATH
else:
    print("⚠️ checkpoint-800 not found — starting from scratch")
    resume = False

print("\n🚀 Starting fine-tuning...")
print("=" * 60)

trainer_stats = trainer.train(resume_from_checkpoint=resume)

print("=" * 60)
print("✅ Training complete!")
print(f"   Total time: {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"   Peak VRAM: {used_memory} GB / {max_memory} GB")

🔁 Resuming from: checkpoint-800
   Remaining steps: ~1075 / 1875

🚀 Starting fine-tuning...


	per_device_train_batch_size: 1 (from args) != 2 (from trainer_state.json)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
825,8.456909
850,9.563821
875,9.553558
900,9.532061
925,9.549263
950,9.571088
975,9.559012
1000,9.545635
1025,9.550400
1050,9.570384


✅ Training complete!
   Total time: 239.2 minutes
   Final loss: 5.4625
   Peak VRAM: 7.617 GB / 14.563 GB


## Step 9: 💾 Save the Fine-Tuned Model

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

LORA_OUTPUT = "/content/drive/MyDrive/cybersec-llama2-lora"
model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)
print(f"✅ LoRA adapters saved to: {LORA_OUTPUT}")

GGUF_OUTPUT = "/content/drive/MyDrive/cybersec-llama2-gguf"
model.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"✅ GGUF model saved to: {GGUF_OUTPUT}")
print(f"   Format: Q4_K_M (recommended for Ollama)")

NameError: name 'model' is not defined

## Step 10: 🧪 Test the Fine-Tuned Model

In [ ]:

from unsloth import FastLanguageModel
import torch

# تحميل النموذج من آخر checkpoint
CHECKPOINT_PATH = "/content/drive/MyDrive/cybersec-llama2-finetuned/checkpoint-1875"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT_PATH,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("✅ Model loaded from checkpoint-1875")

# LoRA مزال محفوظ — skip
print("✅ LoRA already saved — skipping")

# GGUF — تعاود التحويل
GGUF_OUTPUT = "/content/drive/MyDrive/cybersec-llama2-gguf"
model.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"✅ GGUF saved to: {GGUF_OUTPUT}")

## ✅ Done!

| File | Path | Use |
|------|------|-----|
| LoRA adapters | `cybersec-llama2-lora/` | For further training or merging |
| GGUF model | `cybersec-llama2-gguf/` | For Ollama (local deployment) |

### 🏠 To use with Ollama locally:
1. Download `cybersec-llama2-gguf/` from Google Drive
2. Create Modelfile:
   ```
   FROM ./unsloth.Q4_K_M.gguf
   SYSTEM "You are an advanced AI assistant specialized in cybersecurity..."
   ```
3. `ollama create cybersec-assistant -f Modelfile`
4. `ollama run cybersec-assistant`